In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

DATASET_ROOT = "/content/drive/MyDrive/project_dataset"

print("Root contents:", os.listdir(DATASET_ROOT))

print("Wheat classes:", os.listdir(DATASET_ROOT + "/Wheat"))
print("Potato classes:", os.listdir(DATASET_ROOT + "/Potato"))
print("Mango classes:", os.listdir(DATASET_ROOT + "/Mango"))

Root contents: ['Wheat', 'Potato', 'Mango', 'efficientnet_stage1_wheat.h5', 'efficientnet_stage2_potato.h5', 'efficientnet_stage2_potato_finetuned.h5', 'efficientnet_stage3_mango.h5']
Wheat classes: ['aphids', 'armyworms', 'Fusarium head blight', 'Mite', 'Brown Rust', 'root rot', 'Stem fly', 'Healthy', 'Mildew', 'septoria leaf blotch', 'stem rust', 'Yellow Rust', 'wheat blast']
Potato classes: ['aphids', 'blackspot bruising', 'Blackleg', 'Common scab', 'Cutworms', 'Bacterial wilt', 'early_blight', 'Flea Beetle', 'dry rot', 'Healthy', 'potato_tuber_moth', 'PVY', 'late_blight', 'whiteflies', 'soft rot']
Mango classes: ['Anthracnose', 'Bacterial Canker', 'DieBack', 'gall midge', 'Mango Fruit Fly', 'Healthy', 'Mango Mealybug', 'powdery mildew', 'sooty Mold', 'mango_hopper', 'stem-end-rot', 'Thrips', 'Weevil']


In [4]:
import os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [5]:
DATASET_ROOT = "/content/drive/MyDrive/project_dataset"
WHEAT_DIR = os.path.join(DATASET_ROOT, "Wheat")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

In [6]:
WHEAT_CLASSES = sorted(os.listdir(WHEAT_DIR))
N_WHEAT = len(WHEAT_CLASSES)

print("Wheat classes:", WHEAT_CLASSES)
print("Total Wheat classes:", N_WHEAT)

Wheat classes: ['Brown Rust', 'Fusarium head blight', 'Healthy', 'Mildew', 'Mite', 'Stem fly', 'Yellow Rust', 'aphids', 'armyworms', 'root rot', 'septoria leaf blotch', 'stem rust', 'wheat blast']
Total Wheat classes: 13


In [7]:
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=VALIDATION_SPLIT,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    WHEAT_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    WHEAT_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

Found 2207 images belonging to 13 classes.
Found 545 images belonging to 13 classes.


In [8]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.5)(x)
output = Dense(N_WHEAT, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# Freeze pretrained layers first
for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,066,224 (15.51 MB)

 Trainable params: 16,653 (65.05 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [9]:
SAVE_PATH = "/content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5"

checkpoint = ModelCheckpoint(
    SAVE_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    mode="max",
    verbose=1,
    restore_best_weights=True
)

In [10]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[checkpoint, early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.2869 - loss: 2.1880 
Epoch 1: val_accuracy improved from -inf to 0.63670, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5


69/69 ━━━━━━━━━━━━━━━━━━━━ 1533s 22s/step - accuracy: 0.2887 - loss: 2.1828 - val_accuracy: 0.6367 - val_loss: 1.2984
Epoch 2/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6225 - loss: 1.2189
Epoch 2: val_accuracy improved from 0.63670 to 0.67156, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5


69/69 ━━━━━━━━━━━━━━━━━━━━ 302s 4s/step - accuracy: 0.6229 - loss: 1.2178 - val_accuracy: 0.6716 - val_loss: 1.0873
Epoch 3/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7049 - loss: 0.9599
Epoch 3: val_accuracy improved from 0.67156 to 0.70826, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5


69/69 ━━━━━━━━━━━━━━━━━━━━ 296s 4s/step - accuracy: 0.7049 - loss: 0.9596 - val_accuracy: 0.7083 - val_loss: 0.9580
Epoch 4/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7425 - loss: 0.8377
Epoch 4: val_accuracy improved from 0.70826 to 0.73578, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5


69/69 ━━━━━━━━━━━━━━━━━━━━ 310s 4s/step - accuracy: 0.7425 - loss: 0.8373 - val_accuracy: 0.7358 - val_loss: 0.9015
Epoch 5/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7492 - loss: 0.7876
Epoch 5: val_accuracy did not improve from 0.73578
69/69 ━━━━━━━━━━━━━━━━━━━━ 295s 4s/step - accuracy: 0.7494 - loss: 0.7873 - val_accuracy: 0.7321 - val_loss: 0.8819
Epoch 6/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7599 - loss: 0.7116
Epoch 6: val_accuracy improved from 0.73578 to 0.74862, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5


69/69 ━━━━━━━━━━━━━━━━━━━━ 340s 5s/step - accuracy: 0.7601 - loss: 0.7115 - val_accuracy: 0.7486 - val_loss: 0.8304
Epoch 7/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8059 - loss: 0.6734
Epoch 7: val_accuracy did not improve from 0.74862
69/69 ━━━━━━━━━━━━━━━━━━━━ 321s 5s/step - accuracy: 0.8058 - loss: 0.6732 - val_accuracy: 0.7450 - val_loss: 0.8461
Epoch 8/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.8001 - loss: 0.6137
Epoch 8: val_accuracy did not improve from 0.74862
69/69 ━━━━━━━━━━━━━━━━━━━━ 316s 5s/step - accuracy: 0.8001 - loss: 0.6137 - val_accuracy: 0.7119 - val_loss: 0.8363
Epoch 9/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7910 - loss: 0.6489
Epoch 9: val_accuracy did not improve from 0.74862
69/69 ━━━━━━━━━━━━━━━━━━━━ 297s 4s/step - accuracy: 0.7913 - loss: 0.6484 - val_accuracy: 0.7413 - val_loss: 0.8032
Epoch 9: early stopping
Restoring model weights from the end of the best epoch: 6.


In [11]:
loss, acc = model.evaluate(val_gen)
print(f"\nFinal Wheat Validation Accuracy: {acc:.4f}")

print(f"\nModel saved in Google Drive at: {SAVE_PATH}")

18/18 ━━━━━━━━━━━━━━━━━━━━ 73s 4s/step - accuracy: 0.7632 - loss: 0.8304

Final Wheat Validation Accuracy: 0.7541

Model saved in Google Drive at: /content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5


In [12]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
import os

In [13]:
DATASET_ROOT = "/content/drive/MyDrive/project_dataset"
POTATO_DIR = os.path.join(DATASET_ROOT, "Potato")

WHEAT_MODEL_PATH = "/content/drive/MyDrive/project_dataset/efficientnet_stage1_wheat.h5"
SAVE_POTATO_MODEL = "/content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5"

IMG_SIZE = (224, 224)
BATCH = 32
VAL_SPLIT = 0.2
SEED = 42

In [14]:
POTATO_CLASSES = sorted(os.listdir(POTATO_DIR))
N_POTATO = len(POTATO_CLASSES)

print("Potato classes:", POTATO_CLASSES)
print("Number of classes:", N_POTATO)

Potato classes: ['Bacterial wilt', 'Blackleg', 'Common scab', 'Cutworms', 'Flea Beetle', 'Healthy', 'PVY', 'aphids', 'blackspot bruising', 'dry rot', 'early_blight', 'late_blight', 'potato_tuber_moth', 'soft rot', 'whiteflies']
Number of classes: 15


In [15]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    POTATO_DIR,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="training"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    POTATO_DIR,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="validation"
)

Found 3361 files belonging to 15 classes.
Using 2689 files for training.
Found 3361 files belonging to 15 classes.
Using 672 files for validation.


In [16]:
def preprocess(img, label):
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img, label

aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

train_ds = train_ds.map(lambda x, y: (aug(x, training=True), y)).map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

In [17]:
base = EfficientNetB0(include_top=False, weights=None, input_shape=(224, 224, 3))

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.5)(x)
output = Dense(N_POTATO, activation="softmax")(x)

model_potato = Model(inputs=base.input, outputs=output)

In [18]:
wheat_model = tf.keras.models.load_model(WHEAT_MODEL_PATH)

matched = 0
for new_w, old_w in zip(model_potato.weights, wheat_model.weights):
    if new_w.shape == old_w.shape:
        new_w.assign(old_w)
        matched += 1

print(f"Matching weights transferred from Wheat → Potato: {matched}")

Matching weights transferred from Wheat → Potato: 312


In [20]:
model_potato.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n=== Starting Phase 2 Training ===")
model_potato.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)
    ]
)


=== Starting Phase 2 Training ===
Epoch 1/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 1015s 11s/step - accuracy: 0.0716 - loss: 2.8377 - val_accuracy: 0.1295 - val_loss: 2.6740
Epoch 2/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 966s 11s/step - accuracy: 0.1458 - loss: 2.6400 - val_accuracy: 0.2560 - val_loss: 2.4349
Epoch 3/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 937s 11s/step - accuracy: 0.1975 - loss: 2.4891 - val_accuracy: 0.3289 - val_loss: 2.2383
Epoch 4/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 909s 11s/step - accuracy: 0.2966 - loss: 2.2933 - val_accuracy: 0.3452 - val_loss: 2.1205
Epoch 5/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 893s 11s/step - accuracy: 0.3492 - loss: 2.1217 - val_accuracy: 0.3750 - val_loss: 2.0224
Epoch 6/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 935s 11s/step - accuracy: 0.4217 - loss: 1.9700 - val_accuracy: 0.3943 - val_loss: 1.9286
Epoch 7/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 900s 11s/step - accuracy: 0.4679 - loss: 1.8413 - val_accuracy: 0.4152 - val_loss: 1.8448
Epoch 8/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 917s 11s/step - accuracy: 0.51

In [22]:
model_potato.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n=== Starting Phase 2.1 Fine-Tuning ===")
model_potato.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            SAVE_POTATO_MODEL,
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
    ]
)


=== Starting Phase 2.1 Fine-Tuning ===
Epoch 1/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.6889 - loss: 1.0917 
Epoch 1: val_accuracy improved from -inf to 0.58482, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 984s 11s/step - accuracy: 0.6892 - loss: 1.0909 - val_accuracy: 0.5848 - val_loss: 1.2393
Epoch 2/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.7607 - loss: 0.8101 
Epoch 2: val_accuracy improved from 0.58482 to 0.66071, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 905s 11s/step - accuracy: 0.7608 - loss: 0.8095 - val_accuracy: 0.6607 - val_loss: 0.9861
Epoch 3/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.7844 - loss: 0.6493 
Epoch 3: val_accuracy improved from 0.66071 to 0.71726, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 905s 11s/step - accuracy: 0.7845 - loss: 0.6489 - val_accuracy: 0.7173 - val_loss: 0.7880
Epoch 4/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.8352 - loss: 0.5340 
Epoch 4: val_accuracy improved from 0.71726 to 0.76637, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 908s 11s/step - accuracy: 0.8352 - loss: 0.5338 - val_accuracy: 0.7664 - val_loss: 0.6544
Epoch 5/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.8657 - loss: 0.4334 
Epoch 5: val_accuracy improved from 0.76637 to 0.81250, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


...

85/85 ━━━━━━━━━━━━━━━━━━━━ 932s 11s/step - accuracy: 0.9102 - loss: 0.2929 - val_accuracy: 0.8824 - val_loss: 0.3459
Epoch 9/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9134 - loss: 0.2768
Epoch 9: val_accuracy did not improve from 0.88244
85/85 ━━━━━━━━━━━━━━━━━━━━ 887s 10s/step - accuracy: 0.9134 - loss: 0.2767 - val_accuracy: 0.8810 - val_loss: 0.3307
Epoch 10/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9223 - loss: 0.2509
Epoch 10: val_accuracy improved from 0.88244 to 0.91964, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 887s 10s/step - accuracy: 0.9223 - loss: 0.2508 - val_accuracy: 0.9196 - val_loss: 0.2398
Epoch 11/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9078 - loss: 0.2533 
Epoch 11: val_accuracy did not improve from 0.91964
85/85 ━━━━━━━━━━━━━━━━━━━━ 890s 10s/step - accuracy: 0.9079 - loss: 0.2531 - val_accuracy: 0.9167 - val_loss: 0.2501
Epoch 12/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9283 - loss: 0.2180
Epoch 12: val_accuracy did not improve from 0.91964
85/85 ━━━━━━━━━━━━━━━━━━━━ 920s 11s/step - accuracy: 0.9283 - loss: 0.2180 - val_accuracy: 0.9122 - val_loss: 0.2509
Epoch 13/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9407 - loss: 0.1875 
Epoch 13: val_accuracy improved from 0.91964 to 0.92113, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 894s 10s/step - accuracy: 0.9407 - loss: 0.1875 - val_accuracy: 0.9211 - val_loss: 0.2152
Epoch 14/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9517 - loss: 0.1661
Epoch 14: val_accuracy improved from 0.92113 to 0.93452, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 955s 11s/step - accuracy: 0.9516 - loss: 0.1662 - val_accuracy: 0.9345 - val_loss: 0.1903
Epoch 15/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9483 - loss: 0.1723 
Epoch 15: val_accuracy improved from 0.93452 to 0.94643, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 933s 11s/step - accuracy: 0.9483 - loss: 0.1723 - val_accuracy: 0.9464 - val_loss: 0.1727


In [23]:
loss, acc = model_potato.evaluate(val_ds)
print(f"\nFINAL Potato Validation Accuracy: {acc:.4f}")
print("Model saved at:", SAVE_POTATO_MODEL)

21/21 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.9334 - loss: 0.2056

FINAL Potato Validation Accuracy: 0.9464
Model saved at: /content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5


In [24]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
import os

In [25]:
DATASET_ROOT = "/content/drive/MyDrive/project_dataset"
MANGO_DIR = os.path.join(DATASET_ROOT, "Mango")

POTATO_MODEL_PATH = "/content/drive/MyDrive/project_dataset/efficientnet_stage2_potato_final.h5"
SAVE_MANGO_MODEL = "/content/drive/MyDrive/project_dataset/efficientnet_stage3_mango_fast.h5"

IMG_SIZE = (224, 224)
BATCH = 32
VAL_SPLIT = 0.2
SEED = 42

In [26]:
MANGO_CLASSES = sorted(os.listdir(MANGO_DIR))
N_MANGO = len(MANGO_CLASSES)

print("Mango classes:", MANGO_CLASSES)
print("Number of classes:", N_MANGO)

Mango classes: ['Anthracnose', 'Bacterial Canker', 'DieBack', 'Healthy', 'Mango Fruit Fly', 'Mango Mealybug', 'Thrips', 'Weevil', 'gall midge', 'mango_hopper', 'powdery mildew', 'sooty Mold', 'stem-end-rot']
Number of classes: 13


In [28]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    MANGO_DIR,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=True,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    MANGO_DIR,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED
)
def preprocess(img, label):
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img, label

Found 2622 files belonging to 13 classes.
Using 2098 files for training.
Found 2622 files belonging to 13 classes.
Using 524 files for validation.


In [29]:
aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

train_ds = train_ds.map(lambda x, y: (aug(x, training=True), y)).map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

In [30]:
base = EfficientNetB0(include_top=False, weights=None, input_shape=(224, 224, 3))

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.3)(x)
output = Dense(N_MANGO, activation="softmax")(x)

model_mango = Model(inputs=base.input, outputs=output)

In [31]:
potato_model = tf.keras.models.load_model(POTATO_MODEL_PATH)

matched = 0
for new_w, old_w in zip(model_mango.weights, potato_model.weights):
    if new_w.shape == old_w.shape:
        new_w.assign(old_w)
        matched += 1

print(f"Transferred {matched} matching layers from Potato → Mango.")

Transferred 312 matching layers from Potato → Mango.


In [32]:
model_mango.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),  # faster learning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_mango.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,   # fast training
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            SAVE_MANGO_MODEL,
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1
        )
    ]
)

Epoch 1/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.4956 - loss: 1.7266 
Epoch 1: val_accuracy improved from -inf to 0.92939, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage3_mango_fast.h5


66/66 ━━━━━━━━━━━━━━━━━━━━ 789s 11s/step - accuracy: 0.4989 - loss: 1.7182 - val_accuracy: 0.9294 - val_loss: 0.3605
Epoch 2/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9232 - loss: 0.3634 
Epoch 2: val_accuracy improved from 0.92939 to 0.99618, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage3_mango_fast.h5


66/66 ━━━━━━━━━━━━━━━━━━━━ 715s 11s/step - accuracy: 0.9234 - loss: 0.3623 - val_accuracy: 0.9962 - val_loss: 0.0594
Epoch 3/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9674 - loss: 0.1600 
Epoch 3: val_accuracy did not improve from 0.99618
66/66 ━━━━━━━━━━━━━━━━━━━━ 693s 10s/step - accuracy: 0.9674 - loss: 0.1598 - val_accuracy: 0.9962 - val_loss: 0.0297
Epoch 4/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9690 - loss: 0.1195 
Epoch 4: val_accuracy improved from 0.99618 to 0.99809, saving model to /content/drive/MyDrive/project_dataset/efficientnet_stage3_mango_fast.h5


66/66 ━━━━━━━━━━━━━━━━━━━━ 700s 11s/step - accuracy: 0.9692 - loss: 0.1192 - val_accuracy: 0.9981 - val_loss: 0.0175
Epoch 5/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9852 - loss: 0.0652 
Epoch 5: val_accuracy did not improve from 0.99809
66/66 ━━━━━━━━━━━━━━━━━━━━ 753s 11s/step - accuracy: 0.9852 - loss: 0.0651 - val_accuracy: 0.9981 - val_loss: 0.0120
Epoch 6/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9895 - loss: 0.0521
Epoch 6: val_accuracy did not improve from 0.99809
66/66 ━━━━━━━━━━━━━━━━━━━━ 690s 10s/step - accuracy: 0.9895 - loss: 0.0520 - val_accuracy: 0.9981 - val_loss: 0.0091
Epoch 7/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9929 - loss: 0.0373 
Epoch 7: val_accuracy did not improve from 0.99809
66/66 ━━━━━━━━━━━━━━━━━━━━ 695s 11s/step - accuracy: 0.9929 - loss: 0.0372 - val_accuracy: 0.9981 - val_loss: 0.0049
Epoch 8/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9924 - loss: 0.0306
Epoch 8: val_accuracy did not improve fro

66/66 ━━━━━━━━━━━━━━━━━━━━ 754s 11s/step - accuracy: 0.9916 - loss: 0.0324 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 10/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.9947 - loss: 0.0273 
Epoch 10: val_accuracy did not improve from 1.00000
66/66 ━━━━━━━━━━━━━━━━━━━━ 745s 11s/step - accuracy: 0.9947 - loss: 0.0273 - val_accuracy: 1.0000 - val_loss: 0.0024


In [33]:
loss, acc = model_mango.evaluate(val_ds)
print(f"\nFast Mango Validation Accuracy: {acc:.4f}")
print("Model saved at:", SAVE_MANGO_MODEL)

17/17 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 1.0000 - loss: 0.0025

Fast Mango Validation Accuracy: 1.0000
Model saved at: /content/drive/MyDrive/project_dataset/efficientnet_stage3_mango_fast.h5
